In [ ]:
import pandas as pd
import os

RESULTS_DIR = "./data/results"
output_path = "./data/results/q2_q3_analysis.txt"

def log(text=""):
    print(text)
    with open(output_path, "a") as f:
        f.write(text + "\n")

# clear the file first
open(output_path, "w").close()

SCORE_FILES = {
    "LHC":                      "score_deltas_LHC.csv",
    "Gravitational Waves":      "score_deltas_Gravitational_Waves.csv",
    "Complex Network Analysis": "score_deltas_Complex_Network_Analysis.csv",
    "Magnetic Thin Films":      "score_deltas_Magnetic_Thin_Films.csv",
}

log("=" * 60)
log("Q2 — Institutions that flipped from consumer to producer")
log("score = citation_in - citation_out")
log("negative = consumer (cites more than cited)")
log("positive = producer (cited more than cites)")
log("=" * 60)

for domain, fname in SCORE_FILES.items():
    path = os.path.join(RESULTS_DIR, fname)
    df = pd.read_csv(path)

    to_producer = df[(df["score_2005"] < 0) & (df["score_2025"] > 0)].copy()
    to_producer["delta"] = to_producer["score_2025"] - to_producer["score_2005"]
    to_producer = to_producer.sort_values("delta", ascending=False)

    to_consumer = df[(df["score_2005"] > 0) & (df["score_2025"] < 0)].copy()
    to_consumer["delta"] = abs(to_consumer["score_2025"] - to_consumer["score_2005"])
    to_consumer = to_consumer.sort_values("delta", ascending=False)

    log(f"\n{domain}")
    log(f"  Top consumer -> producer transitions:")
    for _, row in to_producer.head(5).iterrows():
        log(f"    {row['name']} ({row['country']}): {row['score_2005']} -> {row['score_2025']}")
    log(f"  Top producer -> consumer transitions:")
    for _, row in to_consumer.head(3).iterrows():
        log(f"    {row['name']} ({row['country']}): {row['score_2005']} -> {row['score_2025']}")

EDGE_FILES = {
    "LHC":                      "reciprocal_edges_LHC.txt",
    "Gravitational Waves":      "reciprocal_edges_Gravitational_Waves.txt",
    "Complex Network Analysis": "reciprocal_edges_Complex_Network_Analysis.txt",
    "Magnetic Thin Films":      "reciprocal_edges_Magnetic_Thin_Films.txt",
}

def parse_reciprocal_edges(path):
    rows = []
    current_year = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("Year:"):
                current_year = int(line.split(":")[1].strip())
            elif "<->" in line and current_year is not None:
                parts = line.split("<->")
                inst1 = parts[0].strip()
                right = parts[1].split(":")
                inst2 = right[0].strip()
                weight = int(right[1].strip())
                rows.append({"year": current_year, "inst1": inst1,
                             "inst2": inst2, "weight": weight})
    return pd.DataFrame(rows)

log("\n" + "=" * 60)
log("Q3 — Strongest reciprocal citation bonds")
log("=" * 60)

for domain, fname in EDGE_FILES.items():
    path = os.path.join(RESULTS_DIR, fname)
    df = parse_reciprocal_edges(path)

    df["pair"] = df.apply(lambda r: tuple(sorted([r["inst1"], r["inst2"]])), axis=1)
    persistence = df.groupby("pair").agg(
        years_present=("year", "count"),
        avg_weight=("weight", "mean"),
        max_weight=("weight", "max"),
    ).sort_values("years_present", ascending=False)

    strongest = df.sort_values("weight", ascending=False).head(1).iloc[0]

    log(f"\n{domain}")
    log(f"  Strongest single bond: {strongest['inst1']} <-> {strongest['inst2']}"
        f" (year {strongest['year']}, weight {strongest['weight']})")
    log(f"  Most persistent pairs:")
    for pair, row in persistence.head(5).iterrows():
        log(f"    {pair[0]} <-> {pair[1]}: "
            f"{int(row['years_present'])} years, "
            f"avg weight {row['avg_weight']:.1f}, "
            f"max weight {int(row['max_weight'])}")

log(f"\nSaved: {output_path}")

Q2 — Institutions that flipped from consumer to producer
score = citation_in - citation_out
negative = consumer (cites more than cited)
positive = producer (cited more than cites)

LHC
  Top consumer -> producer transitions:
    Institut national de recherche en informatique et en automatique (FR): -183 -> 336
    Chinese Academy of Sciences (CN): -408 -> 76
    Chinese University of Hong Kong (CN): -298 -> 77
    Xidian University (CN): -245 -> 113
    University of Hong Kong (HK): -304 -> 41
  Top producer -> consumer transitions:
    University of Chicago (US): 43 -> -82
    Siemens (Germany) (DE): 10 -> -70
    City College of New York (US): 38 -> -34

Gravitational Waves
  Top consumer -> producer transitions:
    University of Maryland, College Park (US): -65 -> 699
    The University of Tokyo (JP): -233 -> 467
    Center for Astrophysics Harvard & Smithsonian (US): -189 -> 441
    West Virginia University (US): -464 -> 145
    Cardiff University (GB): -98 -> 508
  Top producer -